In [ ]:
###############################################
# BMIQ CORRECTION FOR ILLUMINA EPIC PARQUET
# - Uses wateRmelon::BMIQ (canonical implementation)
# - Parallel over samples using future.apply
# - Assumes layout: id_tissue | CpGs... | label
###############################################

# Install packages if needed (run once)
install.packages(c("arrow", "data.table", "future.apply"))
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install("wateRmelon")

library(arrow)         # For Parquet I/O
library(data.table)    # Fast data handling
library(future.apply)  # Parallel apply
library(wateRmelon)    # BMIQ implementation

## -------------------------
## USER SETTINGS (EDIT HERE)
## -------------------------
INPUT_PARQUET  <- "/kaggle/input/3-gse287331-parquet/GSE287331_clean_imputed.parquet" # Input β matrix (sample × CpG)
OUTPUT_PARQUET <- "/kaggle/working/GSE287331_biased_corrected.parquet"                # Output path
MANIFEST_PATH  <- "/kaggle/input/manifest-infinium-methylationepic-cpg-to-gene/infinium-methylationepic-v-1-0-b5-manifest-file.csv" 
ID_COL         <- "id_tissue"                 # sample id column name
LABEL_COL      <- "label"                     # label column name
N_WORKERS      <- max(1, parallel::detectCores() - 1)  # number of parallel workers

## ------------------------------------
## 0) Helper functions for robust manifest loading
## ------------------------------------

detect_separator <- function(sample_line, default = ",") {
  if (grepl("\t", sample_line, fixed = TRUE)) return("\t")
  if (grepl(";",  sample_line, fixed = TRUE)) return(";")
  if (grepl(",",  sample_line, fixed = TRUE)) return(",")
  return(default)
}

open_text_auto <- function(path) {
  if (grepl("\\.gz$", path)) {
    con <- gzfile(path, open = "rt")
  } else if (grepl("\\.bz2$", path)) {
    con <- bzfile(path, open = "rt")
  } else {
    con <- file(path, open = "rt")
  }
  con
}

load_illumina_manifest_any <- function(path,
                                       header_probe_keys = c("IlmnID","Probe_ID","Name","TargetID","ID_REF","CpG"),
                                       fallback_skiprows = 6L) {
  if (!file.exists(path)) {
    stop("[ERROR] Manifest file does not exist: ", path)
  }
  
  con <- open_text_auto(path)
  on.exit(close(con), add = TRUE)
  
  lines <- readLines(con, warn = FALSE)
  if (length(lines) < 8L) {
    stop("[ERROR] Manifest file too short to contain header + data.")
  }
  
  header_idx <- NA_integer_
  
  # 1) Look for [Assay]
  max_scan <- min(200L, length(lines))
  for (i in seq_len(max_scan)) {
    if (startsWith(tolower(trimws(lines[i])), "[assay]")) {
      # search next non-empty, non-[...] line
      upper_j <- min(i + 50L, length(lines))
      for (j in (i + 1L):upper_j) {
        s <- trimws(lines[j])
        if (nchar(s) > 0L && !startsWith(s, "[")) {
          header_idx <- j
          break
        }
      }
      if (!is.na(header_idx)) break
    }
  }
  
  # 2) Fallback: look for known header tokens
  if (is.na(header_idx)) {
    max_scan2 <- min(50L, length(lines))
    for (i in seq_len(max_scan2)) {
      s <- trimws(lines[i])
      parts <- strsplit(s, ",", fixed = TRUE)[[1]]
      if (length(parts) == 1L) {
        parts <- strsplit(s, "\t", fixed = TRUE)[[1]]
      }
      if (length(parts) == 1L) {
        parts <- strsplit(s, ";", fixed = TRUE)[[1]]
      }
      parts <- trimws(parts)
      if (any(parts %in% header_probe_keys)) {
        header_idx <- i
        break
      }
    }
  }
  
  # 3) Ultimate fallback
  if (is.na(header_idx)) {
    header_idx <- fallback_skiprows + 1L  # +1 because fread's header=TRUE expects header line itself
    warning("[manifest] Header not detected → applying fallback skiprows = ", fallback_skiprows)
  }
  
  header_line <- lines[header_idx]
  sep <- detect_separator(header_line, default = ",")
  
  # Build text from header_idx to end
  csv_text <- paste(lines[header_idx:length(lines)], collapse = "\n")
  
  man <- data.table::fread(
    input   = csv_text,
    sep     = sep,
    header  = TRUE,
    data.table = TRUE,
    showProgress = FALSE
  )
  
  # Normalize CpG column name
  cpg_col <- NULL
  for (cand in header_probe_keys) {
    if (cand %in% names(man)) {
      setnames(man, cand, "CpG")
      cpg_col <- "CpG"
      break
    }
  }
  if (is.null(cpg_col)) {
    stop("[ERROR] 'CpG' column not found in manifest (expected one of: ",
         paste(header_probe_keys, collapse = ", "), ").")
  }
  
  man[]
}

## ------------------------------------
## 1) Load Parquet β-matrix (Arrow → DT)
## ------------------------------------
cat("[INFO] Loading Parquet β-matrix from:", INPUT_PARQUET, "\n")
tab <- arrow::read_parquet(INPUT_PARQUET)
dt  <- as.data.table(tab)

cat("[INFO] Loaded matrix with shape: ",
    nrow(dt), "samples ×", ncol(dt) - 2, "CpGs (approx.)\n")

# Identify meta vs CpG columns
meta_cols <- c(ID_COL, LABEL_COL)
if (!all(meta_cols %in% names(dt))) {
  stop("[ERROR] ID_COL and/or LABEL_COL not found in the Parquet file.")
}
cpg_cols <- setdiff(names(dt), meta_cols)

cat("[INFO] Detected", length(cpg_cols), "CpG columns.\n")

## ------------------------------------
## 2) Load EPIC manifest and probe types (ROBUST)
## ------------------------------------
cat("[INFO] Loading EPIC manifest from:", MANIFEST_PATH, "\n")

man_full <- load_illumina_manifest_any(MANIFEST_PATH)

# At this point, man_full has a "CpG" column
if (!("CpG" %in% names(man_full))) {
  stop("[ERROR] Manifest did not contain a 'CpG' column after parsing.")
}

# Detect Infinium design-type column (name can vary)
design_type_col <- NULL
for (cand in c("Infinium_Design_Type",
               "InfiniumDesignType",
               "Design_Type",
               "Infinium_Type",
               "Infinium.Design.Type")) {
  if (cand %in% names(man_full)) {
    design_type_col <- cand
    break
  }
}

if (is.null(design_type_col)) {
  stop("[ERROR] Could not find an Infinium design type column in manifest. ",
       "Looked for: Infinium_Design_Type, InfiniumDesignType, Design_Type, Infinium_Type, Infinium.Design.Type.")
}

cat("[INFO] Using manifest columns: CpG id = CpG, design type =", design_type_col, "\n")

# Keep only manifest rows for CpGs present in the dataset
man_sub <- man_full[ CpG %in% cpg_cols ]

cat("[INFO] Manifest rows matching dataset CpGs:", nrow(man_sub), "\n")

if (nrow(man_sub) == 0L) {
  stop("[ERROR] No CpGs from the manifest matched the dataset columns.")
}

# Build design vector aligned to dataset columns
design_vec_raw <- man_sub[[design_type_col]]
names(design_vec_raw) <- man_sub[["CpG"]]

# Reorder to match the CpG order in the beta matrix
design_vec <- design_vec_raw[cpg_cols]

# Normalize possible variant labels to "I"/"II"
design_vec <- as.character(design_vec)
# Map anything containing "II" → "II", then "I" → "I"
design_vec[grepl("II", design_vec, ignore.case = TRUE)] <- "II"
design_vec[grepl("I",  design_vec, ignore.case = TRUE) & is.na(design_vec) == FALSE & design_vec != "II"] <- "I"

if (any(is.na(design_vec))) {
  n_na <- sum(is.na(design_vec))
  cat("[WARN] There are", n_na, "CpGs without a design type; dropping them.\n")
  keep_idx   <- which(!is.na(design_vec))
  cpg_cols   <- cpg_cols[keep_idx]
  design_vec <- design_vec[keep_idx]
  cat("[INFO] After dropping NA-design CpGs:", length(cpg_cols), "CpGs remain.\n")
}

design_vec <- factor(design_vec, levels = c("I", "II"))
if (any(is.na(design_vec))) {
  stop("[ERROR] Still have NA in design_vec after cleaning. Check manifest mapping.")
}

## ------------------------------------
## 3) Build β matrix (samples × CpGs)
## ------------------------------------
cat("[INFO] Building β matrix (samples × CpGs) in memory…\n")

# Keep only the CpG columns + id/label
beta_mat <- as.matrix(dt[, ..cpg_cols])
rownames(beta_mat) <- dt[[ID_COL]]

# Ensure numeric
storage.mode(beta_mat) <- "double"

cat("[INFO] β matrix dimensions:", nrow(beta_mat), "samples ×",
    ncol(beta_mat), "CpGs\n")

## ------------------------------------
## 4) BMIQ per sample (parallel with future.apply)
## ------------------------------------
cat("[INFO] Starting BMIQ correction in parallel with", N_WORKERS, "workers…\n")

plan(multisession, workers = N_WORKERS)

# Wrap BMIQ in a safe function to catch errors
bmiq_per_sample <- function(bvec, design_v) {
  # bvec: numeric vector of β-values for one sample
  # design_v: factor/character vector with "I"/"II" for each CpG (same length/order as bvec)
  suppressMessages({
    res <- BMIQ(
      beta.v   = as.numeric(bvec),
      design.v = design_v,
      plots    = FALSE
    )
  })
  # res$nbeta is the normalized β vector
  return(res$nbeta)
}

# future_apply over rows (MARGIN = 1)
bmiq_res <- future_apply(
  beta_mat,
  MARGIN = 1,   # per row = per sample
  FUN    = function(b) bmiq_per_sample(b, design_vec)
)

# Convert output to matrix with same orientation as beta_mat
if (is.list(bmiq_res)) {
  cat("[INFO] Converting BMIQ list output to matrix…\n")
  beta_bmiq <- do.call(rbind, bmiq_res)
} else {
  beta_bmiq <- bmiq_res
}

beta_bmiq <- as.matrix(beta_bmiq)
if (!all(dim(beta_bmiq) == dim(beta_mat))) {
  cat("[WARN] Dimension mismatch between beta_mat and beta_bmiq, trying to transpose…\n")
  if (all(rev(dim(beta_bmiq)) == dim(beta_mat))) {
    beta_bmiq <- t(beta_bmiq)
  } else {
    stop("[ERROR] Could not align BMIQ output to input dimensions.")
  }
}

rownames(beta_bmiq) <- rownames(beta_mat)
colnames(beta_bmiq) <- colnames(beta_mat)

cat("[INFO] BMIQ correction completed.\n")

## ------------------------------------
## 5) Reconstruct final table and write Parquet
## ------------------------------------
cat("[INFO] Reconstructing output data.table…\n")

dt_out <- data.table(
  # preserve original id and label
  id_tissue = dt[[ID_COL]],
  label     = dt[[LABEL_COL]]
)

# Add BMIQ-normalized CpGs
dt_out <- cbind(dt_out, as.data.table(beta_bmiq))

cat("[INFO] Writing BMIQ-corrected matrix to:", OUTPUT_PARQUET, "\n")
arrow::write_parquet(
  dt_out,
  sink        = OUTPUT_PARQUET,
  compression = "lz4"
)

cat("[DONE] BMIQ-normalized Parquet saved.\n")


In [1]:
###############################################
# BMIQ CORRECTION FOR ILLUMINA EPIC PARQUET
# - Uses wateRmelon::BMIQ (canonical implementation)
# - Parallel over samples using future.apply
# - Assumes layout: id_tissue | CpGs... | label
# - Optimized for Kaggle 30GB RAM
###############################################

## ------------------------------------
## 0) Package setup (install only if missing)
## ------------------------------------

needed_cran <- c("arrow", "data.table", "future.apply")
for (pkg in needed_cran) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

if (!requireNamespace("wateRmelon", quietly = TRUE)) {
  BiocManager::install("wateRmelon")
}

library(arrow)
library(data.table)
library(future.apply)
library(wateRmelon)

## Allow large globals for future (12 GiB per worker)
options(future.globals.maxSize = 12 * 1024^3)  # 12 GiB

## ------------------------------------
## USER SETTINGS
## ------------------------------------
INPUT_PARQUET  <- "/kaggle/input/3-gse287331-parquet/GSE287331_clean_imputed.parquet"
OUTPUT_PARQUET <- "/kaggle/working/GSE287331_biased_corrected.parquet"
MANIFEST_PATH  <- "/kaggle/input/manifest-infinium-methylationepic-cpg-to-gene/infinium-methylationepic-v-1-0-b5-manifest-file.csv"

ID_COL    <- "id_tissue"
LABEL_COL <- "label"

# Use few workers to limit memory replication
N_WORKERS <- min(3L, max(1L, parallel::detectCores() - 1L))

## ------------------------------------
## Helper functions for robust manifest loading
## ------------------------------------

detect_separator <- function(sample_line, default = ",") {
  if (grepl("\t", sample_line, fixed = TRUE)) return("\t")
  if (grepl(";",  sample_line, fixed = TRUE)) return(";")
  if (grepl(",",  sample_line, fixed = TRUE)) return(",")
  return(default)
}

open_text_auto <- function(path) {
  if (grepl("\\.gz$", path)) {
    con <- gzfile(path, open = "rt")
  } else if (grepl("\\.bz2$", path)) {
    con <- bzfile(path, open = "rt")
  } else {
    con <- file(path, open = "rt")
  }
  con
}

load_illumina_manifest_any <- function(path,
                                       header_probe_keys = c("IlmnID","Probe_ID","Name","TargetID","ID_REF","CpG"),
                                       fallback_skiprows = 6L) {
  if (!file.exists(path)) {
    stop("[ERROR] Manifest file does not exist: ", path)
  }
  
  con <- open_text_auto(path)
  on.exit(close(con), add = TRUE)
  
  lines <- readLines(con, warn = FALSE)
  if (length(lines) < 8L) {
    stop("[ERROR] Manifest file too short to contain header + data.")
  }
  
  header_idx <- NA_integer_
  
  # 1) Look for [Assay]
  max_scan <- min(200L, length(lines))
  for (i in seq_len(max_scan)) {
    if (startsWith(tolower(trimws(lines[i])), "[assay]")) {
      upper_j <- min(i + 50L, length(lines))
      for (j in (i + 1L):upper_j) {
        s <- trimws(lines[j])
        if (nchar(s) > 0L && !startsWith(s, "[")) {
          header_idx <- j
          break
        }
      }
      if (!is.na(header_idx)) break
    }
  }
  
  # 2) Fallback: look for known header tokens
  if (is.na(header_idx)) {
    max_scan2 <- min(50L, length(lines))
    for (i in seq_len(max_scan2)) {
      s <- trimws(lines[i])
      parts <- strsplit(s, ",", fixed = TRUE)[[1]]
      if (length(parts) == 1L) {
        parts <- strsplit(s, "\t", fixed = TRUE)[[1]]
      }
      if (length(parts) == 1L) {
        parts <- strsplit(s, ";", fixed = TRUE)[[1]]
      }
      parts <- trimws(parts)
      if (any(parts %in% header_probe_keys)) {
        header_idx <- i
        break
      }
    }
  }
  
  # 3) Ultimate fallback
  if (is.na(header_idx)) {
    header_idx <- fallback_skiprows + 1L
    warning("[manifest] Header not detected → applying fallback skiprows = ", fallback_skiprows)
  }
  
  header_line <- lines[header_idx]
  sep <- detect_separator(header_line, default = ",")
  
  csv_text <- paste(lines[header_idx:length(lines)], collapse = "\n")
  
  man <- data.table::fread(
    input       = csv_text,
    sep         = sep,
    header      = TRUE,
    data.table  = TRUE,
    showProgress = FALSE
  )
  
  # Normalize CpG column name
  cpg_col <- NULL
  for (cand in header_probe_keys) {
    if (cand %in% names(man)) {
      setnames(man, cand, "CpG")
      cpg_col <- "CpG"
      break
    }
  }
  if (is.null(cpg_col)) {
    stop("[ERROR] 'CpG' column not found in manifest (expected one of: ",
         paste(header_probe_keys, collapse = ", "), ").")
  }
  
  man[]
}

## ------------------------------------
## 1) Load Parquet β-matrix (Arrow → data.table)
## ------------------------------------
cat("[INFO] Loading Parquet β-matrix from:", INPUT_PARQUET, "\n")
tab <- arrow::read_parquet(INPUT_PARQUET)
dt  <- as.data.table(tab)
rm(tab); gc()

cat("[INFO] Loaded matrix with shape: ",
    nrow(dt), "samples ×", ncol(dt) - 2, "CpGs (approx.)\n")

meta_cols <- c(ID_COL, LABEL_COL)
if (!all(meta_cols %in% names(dt))) {
  stop("[ERROR] ID_COL and/or LABEL_COL not found in the Parquet file.")
}
cpg_cols <- setdiff(names(dt), meta_cols)

cat("[INFO] Detected", length(cpg_cols), "CpG columns.\n")

## ------------------------------------
## 2) Load EPIC manifest and probe types
## ------------------------------------
cat("[INFO] Loading EPIC manifest from:", MANIFEST_PATH, "\n")
man_full <- load_illumina_manifest_any(MANIFEST_PATH)

if (!("CpG" %in% names(man_full))) {
  stop("[ERROR] Manifest did not contain a 'CpG' column after parsing.")
}

design_type_col <- NULL
for (cand in c("Infinium_Design_Type",
               "InfiniumDesignType",
               "Design_Type",
               "Infinium_Type",
               "Infinium.Design.Type")) {
  if (cand %in% names(man_full)) {
    design_type_col <- cand
    break
  }
}

if (is.null(design_type_col)) {
  stop("[ERROR] Could not find an Infinium design type column in manifest. ",
       "Looked for: Infinium_Design_Type, InfiniumDesignType, Design_Type, Infinium_Type, Infinium.Design.Type.")
}

cat("[INFO] Using manifest columns: CpG id = CpG, design type =", design_type_col, "\n")

man_sub <- man_full[CpG %in% cpg_cols]
rm(man_full); gc()

cat("[INFO] Manifest rows matching dataset CpGs:", nrow(man_sub), "\n")

if (nrow(man_sub) == 0L) {
  stop("[ERROR] No CpGs from the manifest matched the dataset columns.")
}

design_vec_raw <- man_sub[[design_type_col]]
names(design_vec_raw) <- man_sub[["CpG"]]
rm(man_sub); gc()

design_vec <- design_vec_raw[cpg_cols]
rm(design_vec_raw); gc()

design_vec <- as.character(design_vec)
design_vec[grepl("II", design_vec, ignore.case = TRUE)] <- "II"
design_vec[grepl("I",  design_vec, ignore.case = TRUE) & !is.na(design_vec) & design_vec != "II"] <- "I"

if (any(is.na(design_vec))) {
  n_na <- sum(is.na(design_vec))
  cat("[WARN] There are", n_na, "CpGs without a design type; dropping them.\n")
  keep_idx   <- which(!is.na(design_vec))
  cpg_cols   <- cpg_cols[keep_idx]
  design_vec <- design_vec[keep_idx]
  cat("[INFO] After dropping NA-design CpGs:", length(cpg_cols), "CpGs remain.\n")
}

design_vec <- factor(design_vec, levels = c("I", "II"))
if (any(is.na(design_vec))) {
  stop("[ERROR] Still have NA in design_vec after cleaning. Check manifest mapping.")
}

## ------------------------------------
## 3) Build β matrix (samples × CpGs)
## ------------------------------------
cat("[INFO] Building β matrix (samples × CpGs) in memory…\n")

beta_mat <- as.matrix(dt[, ..cpg_cols])
rownames(beta_mat) <- dt[[ID_COL]]
storage.mode(beta_mat) <- "double"

cat("[INFO] β matrix dimensions:", nrow(beta_mat), "samples ×",
    ncol(beta_mat), "CpGs\n")

## ------------------------------------
## 4) BMIQ per sample (parallel with future.apply)
## ------------------------------------
cat("[INFO] Starting BMIQ correction with", N_WORKERS, "workers…\n")

future::plan(future::multisession, workers = N_WORKERS)

bmiq_per_sample <- function(bvec, design_v) {
  suppressMessages({
    res <- BMIQ(
      beta.v   = as.numeric(bvec),
      design.v = design_v,
      plots    = FALSE
    )
  })
  res$nbeta
}

bmiq_res <- future_apply(
  beta_mat,
  MARGIN = 1,
  FUN    = function(b) bmiq_per_sample(b, design_vec)
)

cat("[INFO] Converting BMIQ output to matrix…\n")
if (is.list(bmiq_res)) {
  beta_bmiq <- do.call(rbind, bmiq_res)
} else {
  beta_bmiq <- bmiq_res
}
rm(bmiq_res); gc()

beta_bmiq <- as.matrix(beta_bmiq)
if (!all(dim(beta_bmiq) == dim(beta_mat))) {
  cat("[WARN] Dimension mismatch between beta_mat and beta_bmiq, trying to transpose…\n")
  if (all(rev(dim(beta_bmiq)) == dim(beta_mat))) {
    beta_bmiq <- t(beta_bmiq)
  } else {
    stop("[ERROR] Could not align BMIQ output to input dimensions.")
  }
}

rownames(beta_bmiq) <- rownames(beta_mat)
colnames(beta_bmiq) <- colnames(beta_mat)

cat("[INFO] BMIQ correction completed.\n")

## ------------------------------------
## 5) Reconstruct final table and write Parquet
## ------------------------------------
cat("[INFO] Reconstructing output data.table…\n")

dt_out <- data.table(
  id_tissue = dt[[ID_COL]],
  label     = dt[[LABEL_COL]]
)
rm(dt); gc()

dt_out <- cbind(dt_out, as.data.table(beta_bmiq))
rm(beta_mat, beta_bmiq); gc()

cat("[INFO] Writing BMIQ-corrected matrix to:", OUTPUT_PARQUET, "\n")
arrow::write_parquet(
  dt_out,
  sink        = OUTPUT_PARQUET,
  compression = "lz4"
)

cat("[DONE] BMIQ-normalized Parquet saved.\n")


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: http://cran.rstudio.com/

Bioconductor version 3.19 (BiocManager 1.30.23), R 4.4.0 (2024-04-24)

Installing package(s) 'wateRmelon'

also installing the dependencies ‘Rhtslib’, ‘multtest’, ‘scrime’, ‘sparseMatrixStats’, ‘SparseArray’, ‘Rsamtools’, ‘GenomicAlignments’, ‘BiocIO’, ‘restfulr’, ‘TxDb.Hsapiens.UCSC.hg19.knownGene’, ‘org.Hs.eg.db’, ‘Biostrings’, ‘bumphunter’, ‘nor1mix’, ‘siggenes’, ‘DelayedMatrixStats’, ‘GEOquery’, ‘DelayedArray’, ‘HDF5Array’, ‘BiocParallel’, ‘UCSC.utils’, ‘GenomeInfoDbData’, ‘XVector’, ‘MatrixGenerics’, ‘S4Arrays’, ‘KEGGREST’, ‘rtracklayer’, ‘affyio’, ‘zlibbioc’, ‘FDb.InfiniumMethylation.hg19’, ‘minfi’, ‘S4Vectors’, ‘IRanges’, ‘GenomeInfoDb’, ‘GenomicRanges’, ‘SummarizedExperiment’, ‘annotate’, ‘genefilter’, ‘AnnotationDbi’, ‘GenomicFeatures’, ‘affy’, ‘preprocessCore’, ‘Biobase’, ‘methylumi’, ‘

[INFO] Loading Parquet β-matrix from: /kaggle/input/3-gse287331-parquet/GSE287331_clean_imputed.parquet 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,13461355,719.0,21829992,1165.9,21829992,1165.9
Vcells,338174173,2580.1,674070168,5142.8,657224853,5014.3


[INFO] Loaded matrix with shape:  446 samples × 703166 CpGs (approx.)
[INFO] Detected 703166 CpG columns.
[INFO] Loading EPIC manifest from: /kaggle/input/manifest-infinium-methylationepic-cpg-to-gene/infinium-methylationepic-v-1-0-b5-manifest-file.csv 


Warning message in data.table::fread(input = csv_text, sep = sep, header = TRUE, :
“Stopped early on line 865920. Expected 52 fields but found 10. Consider fill=TRUE and comment.char=. First discarded non-empty line: <<[Controls],,,,,,,,,>>”


...

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,13508628,721.5,35192121,1879.5,21829992,1165.9
Vcells,344067309,2625.1,674070168,5142.8,657224853,5014.3


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,13508630,721.5,35192121,1879.5,21829992,1165.9
Vcells,344065563,2625.1,674070168,5142.8,657224853,5014.3


[WARN] There are 193 CpGs without a design type; dropping them.
[INFO] After dropping NA-design CpGs: 702973 CpGs remain.
[INFO] Building β matrix (samples × CpGs) in memory…
[INFO] β matrix dimensions: 446 samples × 702973 CpGs
[INFO] Starting BMIQ correction with 3 workers…
[1] "Fitting EM beta mixture to type1 probes"


Warning message in min(beta1.v):
“no non-missing arguments to min; returning Inf”
Warning message in min(beta2.v):
“no non-missing arguments to min; returning Inf”
Warning message in max(beta1.v):
“no non-missing arguments to max; returning -Inf”
Warning message in max(beta2.v):
“no non-missing arguments to max; returning -Inf”


ERROR: Error in sample.int(length(x), size, replace, prob): cannot take a sample larger than the population when 'replace = FALSE'


In [ ]:
###############################################
# SAFE BMIQ CORRECTION (NO REINSTALL)
# - Skips samples with too few non-NA Type I/II probes
# - Uses future.apply with limited workers
###############################################

library(future)
library(future.apply)
library(wateRmelon)
library(arrow)
library(data.table)

# ---- Settings ----
OUTPUT_PARQUET <- "/kaggle/working/GSE287331_BMIQ.parquet"

# beta_mat: samples x CpGs matrix (numeric)
# design_vec: length = ncol(beta_mat), 1=Type I, 2=Type II
# id_tissue: vector of sample IDs, length = nrow(beta_mat)
# label: vector of labels, length = nrow(beta_mat)

# Quick sanity checks
message("[CHECK] beta_mat dim: ", paste(dim(beta_mat), collapse = " x "))
message("[CHECK] length(design_vec): ", length(design_vec))
if (length(design_vec) != ncol(beta_mat)) {
  stop("design_vec length (", length(design_vec),
       ") != ncol(beta_mat) (", ncol(beta_mat), ")")
}

# Limit workers to keep memory under control
plan(multisession, workers = 4)

# ---- Safe BMIQ wrapper ----
bmiq_per_sample_safe <- function(b, design_vec, min_probes = 50L) {
  # b = beta values for ONE sample (numeric vector)
  b <- as.numeric(b)

  if (length(b) != length(design_vec)) {
    stop("In bmiq_per_sample_safe: length(b) != length(design_vec)")
  }

  beta1.v <- b[design_vec == 1L]
  beta2.v <- b[design_vec == 2L]

  n1 <- sum(!is.na(beta1.v))
  n2 <- sum(!is.na(beta2.v))

  # If too few probes of one type, skip BMIQ for this sample
  if (n1 < min_probes || n2 < min_probes) {
    warning(
      "Skipping BMIQ for this sample: not enough non-NA probes (Type I = ",
      n1, ", Type II = ", n2, ")"
    )
    return(b)  # return original beta values (no correction)
  }

  # Run BMIQ
  res <- BMIQ(
    beta.v   = b,
    design.v = design_vec,
    plots    = FALSE
  )

  # Return normalized beta
  res$nbeta
}

# ---- Run BMIQ in parallel over samples ----
message("[INFO] Starting BMIQ correction over samples...")
beta_bmiq <- future_apply(
  beta_mat,
  MARGIN = 1,
  FUN = function(b) bmiq_per_sample_safe(b, design_vec),
  future.seed = TRUE  # parallel-safe RNG
)
message("[INFO] BMIQ completed.")

# beta_bmiq comes back as a matrix (samples x CpGs)
beta_bmiq <- as.matrix(beta_bmiq)
dim(beta_bmiq) <- dim(beta_mat)
colnames(beta_bmiq) <- colnames(beta_mat)

# ---- Rebuild data.table with id_tissue | CpGs... | label ----
if (!exists("id_tissue") || length(id_tissue) != nrow(beta_bmiq)) {
  stop("id_tissue not found or length != nrow(beta_bmiq).")
}
if (!exists("label") || length(label) != nrow(beta_bmiq)) {
  stop("label not found or length != nrow(beta_bmiq).")
}

dt_bmiq <- as.data.table(beta_bmiq)
dt_bmiq[, id_tissue := id_tissue]
dt_bmiq[, label := label]

# Reorder columns: id_tissue | CpGs... | label
setcolorder(dt_bmiq, c("id_tissue", colnames(beta_mat), "label"))

# ---- Save to Parquet ----
message("[INFO] Writing BMIQ-corrected matrix to: ", OUTPUT_PARQUET)
write_parquet(dt_bmiq, OUTPUT_PARQUET)
message("[DONE] BMIQ-corrected dataset saved.")


[CHECK] beta_mat dim: 446 x 702973

[CHECK] length(design_vec): 702973

[INFO] Starting BMIQ correction over samples...



In [6]:
###############################################
# MEMORY-LEAN BMIQ CORRECTION (SEQUENTIAL)
# - No future.apply, no extra big matrix
# - Overwrites beta_mat in place
###############################################

library(wateRmelon)
library(data.table)
library(arrow)

# ---- Settings ----
OUTPUT_PARQUET <- "/kaggle/working/GSE287331_BMIQ.parquet"

# ---- Sanity checks ----
message("[CHECK] beta_mat dim: ", paste(dim(beta_mat), collapse = " x "))
message("[CHECK] length(design_vec): ", length(design_vec))

if (length(design_vec) != ncol(beta_mat)) {
  stop("design_vec length (", length(design_vec),
       ") != ncol(beta_mat) (", ncol(beta_mat), ")")
}
if (!exists("id_tissue") || length(id_tissue) != nrow(beta_mat)) {
  stop("id_tissue not found or length != nrow(beta_mat).")
}
if (!exists("label") || length(label) != nrow(beta_mat)) {
  stop("label not found or length != nrow(beta_mat).")
}

gc()  # prova a liberare qualcosa prima del giretto lungo

# ---- Safe BMIQ wrapper (uguale di prima, ma usato in for-loop) ----
bmiq_per_sample_safe <- function(b, design_vec, min_probes = 50L) {
  b <- as.numeric(b)

  if (length(b) != length(design_vec)) {
    stop("In bmiq_per_sample_safe: length(b) != length(design_vec)")
  }

  beta1.v <- b[design_vec == 1L]
  beta2.v <- b[design_vec == 2L]

  n1 <- sum(!is.na(beta1.v))
  n2 <- sum(!is.na(beta2.v))

  # Se pochi CpG per un tipo, salta BMIQ e tieni i beta originali
  if (n1 < min_probes || n2 < min_probes) {
    warning(
      "Skipping BMIQ for this sample: not enough non-NA probes (Type I = ",
      n1, ", Type II = ", n2, ")"
    )
    return(b)
  }

  res <- BMIQ(
    beta.v   = b,
    design.v = design_vec,
    plots    = FALSE
  )

  res$nbeta
}

# ---- Loop SEQUENZIALE sui campioni (in-place) ----
n_samples <- nrow(beta_mat)
message("[INFO] Starting BMIQ correction (sequential, in-place) on ", n_samples, " samples")

for (i in seq_len(n_samples)) {
  if (i %% 10 == 0 || i == 1 || i == n_samples) {
    message("[INFO] Sample ", i, "/", n_samples)
  }

  b <- beta_mat[i, ]
  beta_mat[i, ] <- bmiq_per_sample_safe(b, design_vec)
}

message("[INFO] BMIQ correction completed for all samples.")

gc()  # di nuovo, prima di costruire la data.table

# ---- Costruisci data.table: id_tissue | CpGs... | label ----
dt_bmiq <- as.data.table(beta_mat)
dt_bmiq[, id_tissue := id_tissue]
dt_bmiq[, label := label]

# Reorder columns: id_tissue | CpGs... | label
setcolorder(dt_bmiq, c("id_tissue", colnames(beta_mat), "label"))

gc()  # un altro piccolo GC prima del write

# ---- Save to Parquet ----
message("[INFO] Writing BMIQ-corrected matrix to: ", OUTPUT_PARQUET)
write_parquet(dt_bmiq, OUTPUT_PARQUET)
message("[DONE] BMIQ-corrected dataset saved.")


[CHECK] beta_mat dim: 446 x 702973

[CHECK] length(design_vec): 702973



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,19481951,1040.5,37859435,2022.0,22128660,1181.8
Vcells,773055317,5898.0,1165550487,8892.5,1165531288,8892.3


[INFO] Starting BMIQ correction (sequential, in-place) on 446 samples

[INFO] Sample 1/446



[1] "Fitting EM beta mixture to type1 probes"
[1] Inf
[1] 0.01239588
[1] 0.0127967
[1] 0.01125013
[1] 0.00928933
[1] "Done"
[1] "Fitting EM beta mixture to type2 probes"
[1] Inf
[1] 0.01530187
[1] 0.01036111
[1] 0.008398501
[1] 0.007339829
[1] "Done"
[1] "Start normalising type 2 probes"
[1] "Finished for sample 1"
[1] "Fitting EM beta mixture to type1 probes"
[1] Inf
[1] 0.01418146
[1] 0.01429867
[1] 0.01212923
[1] 0.00967523
[1] "Done"
[1] "Fitting EM beta mixture to type2 probes"
[1] Inf
[1] 0.01678696
[1] 0.01126705
[1] 0.009028941
[1] 0.007831367
[1] "Done"
[1] "Start normalising type 2 probes"
[1] "Finished for sample 1"
[1] "Fitting EM beta mixture to type1 probes"
[1] Inf
[1] 0.01207199
[1] 0.0125521
[1] 0.01093606
[1] 0.008960768
[1] "Done"
[1] "Fitting EM beta mixture to type2 probes"
[1] Inf
[1] 0.01599073
[1] 0.01045496
[1] 0.008177868
[1] 0.006969131
[1] "Done"
[1] "Start normalising type 2 probes"
[1] "Finished for sample 1"
[1] "Fitting EM beta mixture to type1 probes"
[

[INFO] Sample 10/446



...

[1] "Fitting EM beta mixture to type1 probes"
[1] Inf
[1] 0.0232952
[1] 0.02068093
[1] 0.01692465
[1] 0.01286414
[1] "Done"
[1] "Fitting EM beta mixture to type2 probes"
[1] Inf
[1] 0.01983472
[1] 0.01371559
[1] 0.01070414
[1] 0.008899867
[1] "Done"
[1] "Start normalising type 2 probes"
[1] "Finished for sample 1"


[INFO] BMIQ correction completed for all samples.



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,19499819,1041.5,37859503,2022.0,37859503,2022.0
Vcells,774505337,5909.1,1165550487,8892.5,1165542012,8892.4


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,20203671,1079.0,37859503,2022.0,37859503,2022.0
Vcells,1093644498,8343.9,1398740584,10671.6,1165550077,8892.5


[INFO] Writing BMIQ-corrected matrix to: /kaggle/working/GSE287331_BMIQ.parquet

[DONE] BMIQ-corrected dataset saved.



In [17]:
message("[DONE]")

[DONE]



In [15]:
system("cp /kaggle/working/GSE287331_BMIQ.parquet /kaggle/working/GSE287331_BMIQ_copy.parquet")
system("ls -lh /kaggle/working")


In [16]:
system("ls -lh /kaggle/working")


In [18]:
print(system("ls -lh /kaggle/working", intern = TRUE))


[1] "total 5.8G"                                                          
[2] "-rw-r--r-- 1 root root 2.9G Nov 17 15:55 GSE287331_BMIQ_copy.parquet"
[3] "-rw-r--r-- 1 root root 2.9G Nov 17 15:30 GSE287331_BMIQ.parquet"     


In [22]:
library(arrow)

# Path del file parquet in /kaggle/working
FILE <- "/kaggle/working/GSE287331_BMIQ.parquet"

# Leggi il parquet come Arrow Table (non come tibble)
tab <- read_parquet(FILE, as_data_frame = FALSE)

# ---- Dimensioni ----
dims <- dim(tab)
n_rows <- dims[1]
n_cols <- dims[2]

cat("Rows:", n_rows, "\n")
cat("Columns:", n_cols, "\n\n")

# ---- Nomi colonne ----
all_cols <- names(tab)

first5 <- head(all_cols, 5)
last5  <- tail(all_cols, 5)

cat("First 5 columns:\n")
print(first5)

cat("\nLast 5 columns:\n")
print(last5)


Rows: 446 
Columns: 702975 

First 5 columns:
[1] "id_tissue"  "cg00000109" "cg00000155" "cg00000158" "cg00000165"

Last 5 columns:
[1] "rs939290"  "rs951295"  "rs966367"  "rs9839873" "label"    


446 samples × 703166 CpGs

4_GSE287331_parquet

In [2]:
library(arrow)
library(data.table)

INPUT_PARQUET <- "/kaggle/input/3-gse287331-parquet/GSE287331_clean_imputed.parquet"

# 1) Leggi il parquet
dt <- as.data.table(read_parquet(INPUT_PARQUET))

# 2) Salva id_tissue e label
id_tissue <- dt[["id_tissue"]]
label     <- dt[["label"]]

# 3) Seleziona le colonne CpG
cpg_cols <- setdiff(colnames(dt), c("id_tissue", "label"))

# 4) Costruisci la matrice beta_mat (samples x CpGs)
beta_mat <- as.matrix(dt[, ..cpg_cols])
colnames(beta_mat) <- cpg_cols


In [4]:
###############################################
# BUILD design_vec FROM EPIC MANIFEST
# - design_vec: 1 = Type I, 2 = Type II
# - Aligned to colnames(beta_mat)
###############################################

library(data.table)

MANIFEST_CSV <- "/kaggle/input/manifest-infinium-methylationepic-cpg-to-gene/infinium-methylationepic-v-1-0-b5-manifest-file.csv"

# Controllo base su beta_mat
if (!exists("beta_mat")) {
  stop("beta_mat not found — create beta_mat first (samples x CpGs).")
}

cpg_cols <- colnames(beta_mat)
message("[CHECK] CpG columns in beta_mat: ", length(cpg_cols))

# Leggi manifest EPIC
# Se il file ha header normali va bene così; se avessi linee Illumina da saltare,
# potremmo aggiungere 'skip = 7', ma proviamo prima senza.
mani <- fread(MANIFEST_CSV)

# Trova colonne con ID CpG e design type
id_col <- NULL
if ("IlmnID" %in% names(mani)) {
  id_col <- "IlmnID"
} else if ("Name" %in% names(mani)) {
  id_col <- "Name"
} else {
  stop("No CpG ID column ('IlmnID' or 'Name') found in manifest.")
}

design_col <- NULL
if ("Infinium_Design_Type" %in% names(mani)) {
  design_col <- "Infinium_Design_Type"
} else {
  stop("No 'Infinium_Design_Type' column found in manifest.")
}

message("[INFO] Using manifest ID column: ", id_col)
message("[INFO] Using manifest design column: ", design_col)

# Seleziona solo le CpG che compaiono nel dataset
mani_sub <- mani[get(id_col) %in% cpg_cols, .(CpG = get(id_col),
                                              design = get(design_col))]

message("[INFO] Manifest CpGs overlapping dataset: ", nrow(mani_sub))

# Converte design in 1/2 (Type I / Type II)
mani_sub[, design_num := fifelse(design == "I", 1L,
                                 fifelse(design == "II", 2L, NA_integer_))]

# Allinea al vettore delle colonne di beta_mat
design_vec <- mani_sub[match(cpg_cols, CpG), design_num]

# Alcune CpG potrebbero non avere design definito → gestiamole
n_na <- sum(is.na(design_vec))
if (n_na > 0) {
  message("[WARN] ", n_na, " CpGs have no Infinium_Design_Type in manifest; dropping them.")
  
  keep <- !is.na(design_vec)
  beta_mat <- beta_mat[, keep, drop = FALSE]
  design_vec <- design_vec[keep]
  
  message("[INFO] New beta_mat dim after dropping: ",
          paste(dim(beta_mat), collapse = " x "))
}

# Check final
message("[DONE] design_vec built. Length = ", length(design_vec))
table(design_vec, useNA = "ifany")


[CHECK] CpG columns in beta_mat: 703166

Warning message in fread(MANIFEST_CSV):
“Stopped early on line 8. Expected 27 fields but found 52. Consider fill=TRUE and comment.char=. First discarded non-empty line: <<IlmnID,Name,AddressA_ID,AlleleA_ProbeSeq,AddressB_ID,AlleleB_ProbeSeq,Infinium_Design_Type,Next_Base,Color_Channel,Forward_Sequence,Genome_Build,CHR,MAPINFO,SourceSeq,Strand,UCSC_RefGene_Name,UCSC_RefGene_Accession,UCSC_RefGene_Group,UCSC_CpG_Islands_Name,Relation_to_UCSC_CpG_Island,Phantom4_Enhancers,Phantom5_Enhancers,DMR,450k_Enhancer,HMM_Island,Regulatory_Feature_Name,Regulatory_Feature_Group,GencodeBasicV12_NAME,GencodeBasicV12_Accession,GencodeBasicV12_Group,GencodeCompV12_NAME,GencodeComp>>”


ERROR: Error in eval(expr, envir, enclos): No CpG ID column ('IlmnID' or 'Name') found in manifest.


In [5]:
###############################################
# BUILD design_vec FROM EPIC MANIFEST (FIXED)
# - design_vec: 1 = Type I, 2 = Type II
# - Aligned to colnames(beta_mat)
###############################################

library(data.table)

MANIFEST_CSV <- "/kaggle/input/manifest-infinium-methylationepic-cpg-to-gene/infinium-methylationepic-v-1-0-b5-manifest-file.csv"

# Controllo base su beta_mat
if (!exists("beta_mat")) {
  stop("beta_mat not found — create beta_mat first (samples x CpGs).")
}

cpg_cols <- colnames(beta_mat)
message("[CHECK] CpG columns in beta_mat: ", length(cpg_cols))

# ---- LEGGI IL MANIFEST IN MODO ROBUSTO ----
# Salta tutte le righe prima di quella che contiene 'IlmnID'
mani <- fread(MANIFEST_CSV, skip = "IlmnID")

message("[INFO] Manifest columns: ", paste(names(mani), collapse = ", "))

# Trova colonne con ID CpG e design type
id_col <- NULL
if ("IlmnID" %in% names(mani)) {
  id_col <- "IlmnID"
} else if ("Name" %in% names(mani)) {
  id_col <- "Name"
} else {
  stop("No CpG ID column ('IlmnID' or 'Name') found in manifest.")
}

design_col <- NULL
if ("Infinium_Design_Type" %in% names(mani)) {
  design_col <- "Infinium_Design_Type"
} else {
  stop("No 'Infinium_Design_Type' column found in manifest.")
}

message("[INFO] Using manifest ID column: ", id_col)
message("[INFO] Using manifest design column: ", design_col)

# Seleziona solo le CpG che compaiono nel dataset
mani_sub <- mani[get(id_col) %in% cpg_cols,
                 .(CpG = get(id_col),
                   design = get(design_col))]

message("[INFO] Manifest CpGs overlapping dataset: ", nrow(mani_sub))

# Converte design in 1/2 (Type I / Type II)
mani_sub[, design_num := fifelse(design == "I", 1L,
                                 fifelse(design == "II", 2L, NA_integer_))]

# Allinea al vettore delle colonne di beta_mat
design_vec <- mani_sub[match(cpg_cols, CpG), design_num]

# Alcune CpG potrebbero non avere design definito → gestiamole
n_na <- sum(is.na(design_vec))
if (n_na > 0) {
  message("[WARN] ", n_na, " CpGs have no Infinium_Design_Type in manifest; dropping them.")
  
  keep <- !is.na(design_vec)
  beta_mat <- beta_mat[, keep, drop = FALSE]
  design_vec <- design_vec[keep]
  cpg_cols <- colnames(beta_mat)
  
  message("[INFO] New beta_mat dim after dropping: ",
          paste(dim(beta_mat), collapse = " x "))
}

# Check finale
message("[DONE] design_vec built. Length = ", length(design_vec))
print(table(design_vec, useNA = "ifany"))


[CHECK] CpG columns in beta_mat: 703166

Warning message in fread(MANIFEST_CSV, skip = "IlmnID"):
“Stopped early on line 865927. Expected 52 fields but found 10. Consider fill=TRUE and comment.char=. First discarded non-empty line: <<[Controls],,,,,,,,,>>”
[INFO] Manifest columns: IlmnID, Name, AddressA_ID, AlleleA_ProbeSeq, AddressB_ID, AlleleB_ProbeSeq, Infinium_Design_Type, Next_Base, Color_Channel, Forward_Sequence, Genome_Build, CHR, MAPINFO, SourceSeq, Strand, UCSC_RefGene_Name, UCSC_RefGene_Accession, UCSC_RefGene_Group, UCSC_CpG_Islands_Name, Relation_to_UCSC_CpG_Island, Phantom4_Enhancers, Phantom5_Enhancers, DMR, 450k_Enhancer, HMM_Island, Regulatory_Feature_Name, Regulatory_Feature_Group, GencodeBasicV12_NAME, GencodeBasicV12_Accession, GencodeBasicV12_Group, GencodeCompV12_NAME, GencodeCompV12_Accession, GencodeCompV12_Group, DNase_Hypersensitivity_NAME, DNase_Hypersensitivity_Evidence_Count, OpenChromatin_NAME, OpenChromatin_Evidence_Count, TFBS_NAME, TFBS_Evidence_Count, 

design_vec
     1      2 
115723 587250 
